#### ***RAG Evaluation***

##### ***The Rag Evaluation the quality of generated RAG Answer.***

#### ***1.Faithfulness***

##### **Does the answer contain information supported by the Retrieved Context.**

#### ***2.Answer Relevance***
##### **Does the answer actually answer the user's question**

#### ***3.Context Relevance***
##### **Is the retrieved Context Relevant to the user question.**

In [1]:
# Load environment variable
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001712E01B980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001717E5B94F0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
## Load the vector store from Chroma
from langchain_chroma import Chroma
vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name = "kubernetes_rag",
    embedding_function=embedding_model
)

In [5]:
### Similarity search
retriever = vectorstore.as_retriever(search_kwargs = {"k":3})

In [11]:
##Design a Prompt
from langchain_core.prompts import ChatPromptTemplate
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template("""
You are a Kubernetes documentation Assistant.

Answer the question using only the provided Context only.


Rules:
    1.Don't use information outside the Context.
    2.If the answer is not available in the context,
    say I don't have information based on the Provided Documents

Context:
{context}

Question:
{question}

Answer:


Evaluate the following:
Generation Evaluation

1.Faithfullness:
Does the answer contain information that have supported by retrieved context.

2.Answer Relevance:
Does the answer actual answer's the user query.

3.Context Relevance:
Is the Retrieved Context Relevant to the User's Question.

give the score from 0 to 1 for each metric.

Return only a valid json format for each metric

""")

In [12]:
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
rag_chain = ({
    "context":retriever | RunnableLambda(format_docs),
    "question":RunnablePassthrough()
}
|prompt
| llm 
| StrOutputParser()
)

In [13]:
test_queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?"
]

In [15]:
for query in test_queries:
    response = rag_chain.invoke(query)
    print("Query:",query)
    print(response)
    print("#"*70)

Query: What is a Kubernetes Deployment?
A Kubernetes Deployment is an API object that represents an application running on a cluster. It defines the desired state (spec) for that application—such as the number of replica Pods—and the Kubernetes system works to make the actual state (status) match the spec, creating, updating, or replacing Pods as needed.

```json
{
  "Faithfullness": 1,
  "Answer Relevance": 1,
  "Context Relevance": 1
}
```
######################################################################
Query: What is a Kubernetes Pod?
**What is a Kubernetes Pod?**

A Kubernetes Pod is the smallest deployable unit in Kubernetes that represents a group of one or more containers (such as Docker containers) which share storage, network resources, and a specification for how to run the containers. The containers in a Pod are always co‑located, co‑scheduled, and run in a shared context, effectively modeling an application‑specific “logical host.” Pods can also include init container